# 1차 실행 -  논문 목록 다운로드

In [ ]:
import os
import requests
import pandas as pd
import xml.etree.ElementTree as ET
import time
from tqdm import tqdm
from dotenv import load_dotenv

# 1. 명시적으로 .env 파일 로드 (파일명 확인 필수!)
if os.path.exists('.env'):
    load_dotenv('.env', override=True) # 기존 환경변수보다 .env 파일 내용을 우선함
else:
    raise FileNotFoundError(".env 파일이 없습니다. .env.sample을 복사해서 .env를 만들어주세요.")

API_KEY = os.getenv("KCI_API_KEY")
BASE_URL = "https://open.kci.go.kr/po/openapi/openApiSearch.kci"

# 키 값이 샘플 값인지 확인하는 안전장치
if not API_KEY or "여기에" in API_KEY:
    raise ValueError("API_KEY가 설정되지 않았거나 샘플 값입니다. .env 파일을 확인하세요.")

def fetch_articles_by_journal(api_key, journal_name):
    articles = []
    page = 1
    display_count = 100
    
    while True:
        params = {
            "apiCode": "articleSearch",
            "key": api_key,
            "journal": journal_name,
            "displayCount": display_count,
            "page": page
        }
        
        try:
            response = requests.get(BASE_URL, params=params, timeout=30)
            response.raise_for_status()
            root = ET.fromstring(response.content)

            # [핵심] 결과 메시지 확인 (키 오류 등 확인용)
            result_msg_node = root.find(".//{*}resultMsg")
            if result_msg_node is not None and "정상" not in result_msg_node.text:
                # '등록되지 않은 key입니다' 등이 여기서 걸러짐
                print(f"\n[API 메시지 - {journal_name}]: {result_msg_node.text}")
                break

            # 결과 개수 확인
            total_node = root.find(".//{*}total")
            if total_node is None or int(total_node.text) == 0:
                break
                
            total_count = int(total_node.text)
            
            # 레코드 추출 시작
            records = root.findall(".//{*}record")
            for record in records:
                try:
                    article_info = record.find("{*}articleInfo")
                    journal_info = record.find("{*}journalInfo")
                    if article_info is None: continue

                    # 데이터 파싱 시 None 방지 (findtext 활용)
                    articles.append({
                        "학술지명": journal_name,
                        "논문ID": article_info.get("article-id", ""),
                        "제목": article_info.findtext(".//{*}article-title", default="제목 없음").strip(),
                        "저자": ", ".join([a.text for a in article_info.findall(".//{*}author") if a.text]),
                        "발행연도": journal_info.findtext("{*}pub-year", default="") if journal_info is not None else "",
                        "KCI_URL": article_info.findtext("{*}url", default="")
                    })
                except Exception:
                    continue # 개별 레코드 오류는 건너뜀
            
            if page * display_count >= total_count:
                break
            page += 1
            time.sleep(0.2) # 적절한 딜레이

        except Exception as e:
            print(f"\n[오류] {journal_name} 수집 중단: {e}")
            break
            
    return articles
# ==========================================
# 실행부
# ==========================================
if __name__ == "__main__":
    INPUT_FILE = '법학 기관 목록.csv'
    OUTPUT_FILE = '전체_법학_논문목록.csv'

    try:
        # 1. 데이터 로드
        df_org = pd.read_csv(INPUT_FILE, encoding='utf-8')
        if '학술지한글명' not in df_org.columns:
            raise ValueError("CSV 파일에 '학술지한글명' 칼럼이 없습니다.")
            
        target_journals = df_org['학술지한글명'].dropna().unique().tolist()
        print(f"총 {len(target_journals)}개의 학술지를 수집합니다.")

        # 2. 수집 진행
        all_results = []
        for jnl in tqdm(target_journals, desc="수집 진행률"):
            results = fetch_articles_by_journal(API_KEY, jnl)
            all_results.extend(results)
            
            # 중간 저장
            if len(all_results) % 500 == 0:
                pd.DataFrame(all_results).to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

        # 3. 최종 저장
        if all_results:
            
            df_final = pd.DataFrame(all_results)
            df_final.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
            print(f"\n✅ 수집 완료! 총 {len(df_final)}건 저장됨.")
        else:
            print("\n❌ 수집된 데이터가 없습니다. API 키와 학술지명을 다시 확인하세요.")

    except Exception as e:
        print(f"프로그램 실행 중 치명적 오류: {e}")

### 초록 포함해서 수집

In [ ]:
import os
import requests
import pandas as pd
import xml.etree.ElementTree as ET
import time
from tqdm import tqdm
from dotenv import load_dotenv

# 1. .env 파일 로드
if os.path.exists('.env'):
    load_dotenv('.env', override=True)
else:
    raise FileNotFoundError(".env 파일이 없습니다.")

API_KEY = os.getenv("KCI_API_KEY")
BASE_URL = "https://open.kci.go.kr/po/openapi/openApiSearch.kci"

def fetch_articles_by_journal(api_key, journal_name):
    articles = []
    page = 1
    display_count = 100
    
    while True:
        params = {
            "apiCode": "articleSearch",
            "key": api_key,
            "journal": journal_name,
            "displayCount": display_count,
            "page": page
        }
        
        try:
            response = requests.get(BASE_URL, params=params, timeout=30)
            response.raise_for_status()
            root = ET.fromstring(response.content)

            result_msg_node = root.find(".//{*}resultMsg")
            if result_msg_node is not None and "정상" not in result_msg_node.text:
                print(f"\n[API 메시지 - {journal_name}]: {result_msg_node.text}")
                break

            total_node = root.find(".//{*}total")
            if total_node is None or int(total_node.text) == 0:
                break
                
            total_count = int(total_node.text)
            records = root.findall(".//{*}record")
            
            for record in records:
                try:
                    article_info = record.find("{*}articleInfo")
                    journal_info = record.find("{*}journalInfo")
                    if article_info is None: continue

                    # --- [초록 추출 로직 시작] ---
                    abstract_text = ""
                    # abstract-group 내의 모든 abstract 태그를 찾음
                    abstract_nodes = article_info.findall(".//{*}abstract")
                    
                    for ab in abstract_nodes:
                        # lang 속성이 'original'인 것을 우선적으로 찾음
                        if ab.get('lang') == 'original':
                            abstract_text = ab.text.strip() if ab.text else ""
                            break
                    
                    # 만약 lang="original"이 없다면 첫 번째 초록이라도 가져옴
                    if not abstract_text and abstract_nodes:
                        abstract_text = abstract_nodes[0].text.strip() if abstract_nodes[0].text else ""
                    # --- [초록 추출 로직 종료] ---

                    articles.append({
                        "학술지명": journal_name,
                        "논문ID": article_info.get("article-id", ""),
                        "제목": article_info.findtext(".//{*}article-title", default="제목 없음").strip(),
                        "저자": ", ".join([a.text for a in article_info.findall(".//{*}author") if a.text]),
                        "초록": abstract_text,  # 신규 추가 칼럼
                        "발행연도": journal_info.findtext("{*}pub-year", default="") if journal_info is not None else "",
                        "KCI_URL": article_info.findtext("{*}url", default="")
                    })
                except Exception:
                    continue 
            
            if page * display_count >= total_count:
                break
            page += 1
            time.sleep(0.2)

        except Exception as e:
            print(f"\n[오류] {journal_name} 수집 중단: {e}")
            break
            
    return articles

if __name__ == "__main__":
    INPUT_FILE = '법학 기관 목록.csv'
    OUTPUT_FILE = '전체_법학_논문목록.csv'

    try:
        df_org = pd.read_csv(INPUT_FILE, encoding='utf-8')
        if '학술지한글명' not in df_org.columns:
            raise ValueError("CSV 파일에 '학술지한글명' 칼럼이 없습니다.")
            
        target_journals = df_org['학술지한글명'].dropna().unique().tolist()
        print(f"총 {len(target_journals)}개의 학술지에서 초록을 포함하여 수집합니다.")

        all_results = []
        for jnl in tqdm(target_journals, desc="수집 진행률"):
            results = fetch_articles_by_journal(API_KEY, jnl) 
            all_results.extend(results)
            
            # 중간 저장 (500건 단위)
            if len(all_results) > 0 and len(all_results) % 500 == 0:
                pd.DataFrame(all_results).to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

        if all_results:
            df_final = pd.DataFrame(all_results)
            df_final.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
            print(f"\n✅ 수집 완료! 초록 포함 총 {len(df_final)}건 저장됨.")
        else:
            print("\n❌ 수집된 데이터가 없습니다.")

    except Exception as e:
        print(f"프로그램 실행 중 치명적 오류: {e}")

### 디버그용 셀(실행 불필요)

In [ ]:
# import os
# import requests
# from dotenv import load_dotenv

# load_dotenv()
# API_KEY = os.getenv("KCI_API_KEY")
# BASE_URL = "https://open.kci.go.kr/po/openapi/openApiSearch.kci"

# # 테스트할 학술지명 하나 지정
# test_journal = "법학연구" 

# params = {
#     "apiCode": "articleSearch",
#     "key": API_KEY,
#     "journal": test_journal,
#     "displayCount": 1,
#     "page": 1
# }

# try:
#     response = requests.get(BASE_URL, params=params)
#     print("="*50)
#     print(f"HTTP 상태 코드: {response.status_code}")
#     print("="*50)
#     # 응답 내용 전체 출력
#     print(response.text)
#     print("="*50)
# except Exception as e:
#     print(f"요청 중 오류 발생: {e}")

# 중복제거

In [ ]:
import pandas as pd

INPUT_FILE = '전체_법학_논문목록.csv'
CLEANED_FILE = '전체_법학_논문목록_중복제거.csv'

try:
    # 1. 데이터 로드
    df = pd.read_csv(INPUT_FILE)
    total_rows = len(df)
    
    # 2. '논문ID' 기준 중복 체크
    duplicate_mask = df.duplicated(subset=['논문ID'], keep=False)
    df_duplicates = df[duplicate_mask].sort_values(by='논문ID')
    
    # 3. 통계 계산
    unique_count = df['논문ID'].nunique()
    duplicate_count = total_rows - unique_count
    # 중복률 계산 (필요시)
    # $Duplication Rate = \frac{N_{duplicates}}{N_{total}} \times 100$
    
    print("="*50)
    print(f"📊 데이터 중복 체크 결과")
    print("-"*50)
    print(f" 전체 행(Row) 수: {total_rows}건")
    print(f" 고유 논문(Unique ID) 수: {unique_count}건")
    print(f" 중복된 데이터 수: {duplicate_count}건")
    print(f" 중복률: {(duplicate_count / total_rows * 100):.2f}%")
    print("="*50)

    # 4. 중복된 논문 샘플 출력 (중복이 있을 경우)
    if duplicate_count > 0:
        print("\n⚠️ 중복된 논문 샘플 (상위 5건):")
        print(df_duplicates[['논문ID', '제목', '학술지명']].head(10))
        
        # 5. 중복 제거 및 저장 선택
        # keep='first'를 써서 중복 중 첫 번째 행만 남김
        df_cleaned = df.drop_duplicates(subset=['논문ID'], keep='first')
        df_cleaned.to_csv(CLEANED_FILE, index=False, encoding='utf-8-sig')
        print(f"\n✅ 중복이 제거된 파일이 '{CLEANED_FILE}'로 저장되었습니다.")
    else:
        print("\n✅ 중복된 논문이 없습니다. 깨끗한 데이터입니다!")

except Exception as e:
    print(f"오류 발생: {e}")

# 세부 참고문헌 정보 다운로드

In [ ]:
import os
import requests
import pandas as pd
import time
import re
from tqdm import tqdm
from dotenv import load_dotenv

# 설정 로드
load_dotenv()
API_KEY = os.getenv("KCI_API_KEY")
BASE_URL = "https://open.kci.go.kr/po/openapi/openApiSearch.kci"

INPUT_FILE = '전체_법학_논문목록.csv'
OUTPUT_FILE = '법학_인용_네트워크_데이터_전체.csv'
SAVE_INTERVAL = 50  # 50건마다 파일에 저장

def fetch_references_ultimate(article_id):
    references = []
    params = {"apiCode": "articleDetail", "key": API_KEY, "id": article_id}
    headers = {"User-Agent": "Mozilla/5.0"}
    
    try:
        response = requests.get(BASE_URL, params=params, headers=headers, timeout=30)
        response.encoding = 'utf-8'
        raw_xml = response.text
        
        if "<total>0</total>" in raw_xml:
            return [], "No Data"

        # 정규표현식 추출
        ref_matches = re.findall(r'<reference([^>]*)>(?:<!\[CDATA\[)?(.*?)(?:\]\]>)?</reference>', raw_xml, re.DOTALL)
        
        for attrs, content in ref_matches:
            arti_id_match = re.search(r'arti-id="([^"]*)"', attrs)
            type_name_match = re.search(r'type-name="([^"]*)"', attrs)
            
            references.append({
                "source_id": article_id,
                "target_arti_id": arti_id_match.group(1) if arti_id_match else "",
                "ref_type": type_name_match.group(1) if type_name_match else "",
                "raw_citation": content.strip()
            })
        return references, "Success"
    except Exception as e:
        return [], str(e)

# 1. 이어하기 로직: 이미 수집된 ID 확인
processed_ids = set()
if os.path.exists(OUTPUT_FILE):
    try:
        df_existing = pd.read_csv(OUTPUT_FILE)
        processed_ids = set(df_existing['source_id'].unique())
        print(f"✅ 이미 수집된 논문 {len(processed_ids)}건을 발견했습니다. 이어서 진행합니다.")
    except:
        pass

# 2. 대상 목록 로드 및 필터링 (필터 제거 버전)
df_articles = pd.read_csv(INPUT_FILE)

# 필터링 없이 전체 논문 ID를 가져옵니다.
article_ids = [aid for aid in df_articles['논문ID'].dropna().unique() if aid not in processed_ids]

print(f"📊 수집 대상: 총 {len(article_ids)}건 (전체 기간)")


# 3. 메인 루프
all_results = []
try:
    for i, aid in enumerate(tqdm(article_ids, desc="전체 수집 진행 중")):
        refs, status = fetch_references_ultimate(aid)
        
        if refs:
            all_results.extend(refs)
        
        # 주기적으로 파일 저장 (메모리 관리 및 데이터 보호)
        if (i + 1) % SAVE_INTERVAL == 0:
            header = not os.path.exists(OUTPUT_FILE)
            pd.DataFrame(all_results).to_csv(OUTPUT_FILE, mode='a', index=False, header=header, encoding='utf-8-sig')
            all_results = [] # 메모리 비우기
            
        time.sleep(1.2) # 서버 부하 방지용 지연

except KeyboardInterrupt:
    print("\n🛑 사용자에 의해 중단되었습니다. 현재까지의 데이터를 저장합니다.")

# 4. 남은 데이터 최종 저장
if all_results:
    header = not os.path.exists(OUTPUT_FILE)
    pd.DataFrame(all_results).to_csv(OUTPUT_FILE, mode='a', index=False, header=header, encoding='utf-8-sig')

print(f"🏁 수집 완료! 결과가 '{OUTPUT_FILE}'에 저장되었습니다.")

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("법학_인용_네트워크_데이터_전체.csv")

In [ ]:
print(len(df["source_id"].unique()))

# 인공지능+특정 연도 수집

In [1]:
"""
KCI 논문 상세정보 및 인용 데이터 수집 스크립트
================================================
KCI OpenAPI를 통해 논문별 키워드·초록·학술분류·인용 정보를 수집.

출력 컬럼:
    source_id, title, pub_year, abstract, keywords, category,
    target_arti_id, ref_type

사용법:
    python 00.fetch_kci_details.py
"""

import os
import re
import time

import pandas as pd
import requests
from dotenv import load_dotenv
from tqdm import tqdm

# ─────────────────────────────────────────────
# ⚙️  CONFIG
# ─────────────────────────────────────────────

load_dotenv()
API_KEY  = os.getenv("KCI_API_KEY")
BASE_URL = "https://open.kci.go.kr/po/openapi/openApiSearch.kci"

INPUT_FILE     = "KCI_AI_논문_기본정보_목록_201601_202512.csv"
OUTPUT_FILE    = "00.KCI_인공_논문_상세_및_인용데이터.csv"
SAVE_INTERVAL  = 50
YEAR_START     = 2016
YEAR_END       = 2025
KEYWORD_FILTER = r"인공지능|AI"


# ─────────────────────────────────────────────
# 🔧 API 호출 및 파싱
# ─────────────────────────────────────────────

def fetch_article_detail_and_refs(article_id: str, title: str, year) -> tuple[list, str]:
    """
    KCI API에서 논문 상세정보(키워드·초록·분류)와 인용 목록을 가져옴.

    Returns:
        (rows, status_message)
        rows: 인용 수만큼의 행 리스트 (인용 없으면 1행)
    """
    params  = {"apiCode": "articleDetail", "key": API_KEY, "id": article_id}
    headers = {"User-Agent": "Mozilla/5.0"}

    try:
        response = requests.get(BASE_URL, params=params, headers=headers, timeout=30)
        response.encoding = "utf-8"
        raw_xml = response.text

        if "<total>0</total>" in raw_xml:
            return [], "No Data"

        # ── 키워드 추출 ──────────────────────────────
        # 버그 수정: \]\]Task → \]\]>
        keyword_group = re.search(r"<keyword-group>(.*?)</keyword-group>", raw_xml, re.DOTALL)
        if keyword_group:
            kw_list = re.findall(
                r"<keyword[^>]*>(?:<!\[CDATA\[)?(.*?)(?:\]\]>)?</keyword>",
                keyword_group.group(1)
            )
            keywords_str = ", ".join([k.strip() for k in kw_list if k.strip()])
        else:
            keywords_str = ""

        # ── 초록 추출 (regex 방식으로 통일) ──────────
        # article_info.findall() 제거 → re.search 로 대체
        abstract_match = re.search(
            r'<abstract[^>]*lang="original"[^>]*>(?:<!\[CDATA\[)?(.*?)(?:\]\]>)?</abstract>',
            raw_xml, re.DOTALL
        )
        if not abstract_match:
            abstract_match = re.search(
                r"<abstract[^>]*>(?:<!\[CDATA\[)?(.*?)(?:\]\]>)?</abstract>",
                raw_xml, re.DOTALL
            )
        abstract_text = abstract_match.group(1).strip() if abstract_match else ""

        # ── 학술분류 추출 ────────────────────────────
        category_match = re.search(
            r"<article-categories[^>]*>(?:<!\[CDATA\[)?(.*?)(?:\]\]>)?</article-categories>",
            raw_xml, re.DOTALL
        )
        category = category_match.group(1).strip() if category_match else ""

        # ── 인용 정보 추출 ───────────────────────────
        ref_matches = re.findall(
            r"<reference([^>]*)>(?:<!\[CDATA\[)?(.*?)(?:\]\]>)?</reference>",
            raw_xml, re.DOTALL
        )

        # 모든 행에 공통으로 들어갈 기본 데이터
        base_data = {
            "source_id": article_id,
            "title":     title,
            "pub_year":  year,
            "abstract":  abstract_text,
            "keywords":  keywords_str,
            "category":  category,
        }

        results = []
        if not ref_matches:
            row = base_data.copy()
            row.update({"target_arti_id": "", "ref_type": ""})
            results.append(row)
        else:
            for attrs, _ in ref_matches:
                arti_id_match   = re.search(r'arti-id="([^"]*)"',   attrs)
                type_name_match = re.search(r'type-name="([^"]*)"', attrs)
                row = base_data.copy()
                row.update({
                    "target_arti_id": arti_id_match.group(1)   if arti_id_match   else "",
                    "ref_type":       type_name_match.group(1) if type_name_match else "",
                })
                results.append(row)

        return results, "Success"

    except Exception as e:
        return [], str(e)


# ─────────────────────────────────────────────
# 🚀 메인
# ─────────────────────────────────────────────

def main():
    # 1. 이어하기 — 이미 수집된 ID 로드
    processed_ids = set()
    if os.path.exists(OUTPUT_FILE):
        try:
            df_existing   = pd.read_csv(OUTPUT_FILE, encoding="utf-8-sig")
            processed_ids = set(df_existing["source_id"].astype(str).unique())
            print(f"[이어하기] 기존 파일 로드: {len(processed_ids)}건 이미 수집됨")
        except Exception:
            pass

    # 2. 대상 로드 및 필터링
    print(f"[로딩] {INPUT_FILE}")
    df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")
    df["발행연도"] = pd.to_numeric(df["발행연도"], errors="coerce")

    year_filter    = (df["발행연도"] >= YEAR_START) & (df["발행연도"] <= YEAR_END)
    keyword_filter = df["제목"].str.contains(KEYWORD_FILTER, case=False, na=False)

    df_filtered   = df[year_filter & keyword_filter]
    df_to_process = df_filtered[
        ~df_filtered["논문ID"].astype(str).isin(processed_ids)
    ].copy()

    print(f"\n{'='*50}")
    print(f"  필터: '{KEYWORD_FILTER}' | {YEAR_START}~{YEAR_END}년")
    print(f"  조건 일치:       {len(df_filtered):>6}건")
    print(f"  이미 수집 제외:  {len(processed_ids):>6}건")
    print(f"  최종 수집 대상:  {len(df_to_process):>6}건")
    print(f"{'='*50}\n")

    # 3. 메인 수집 루프
    all_results = []
    try:
        for i, row in enumerate(
            tqdm(df_to_process.itertuples(index=False),
                 total=len(df_to_process), desc="수집 중", unit="건")
        ):
            aid   = str(row.논문ID)
            title = getattr(row, "제목", "")
            year  = getattr(row, "발행연도", "")

            details, status = fetch_article_detail_and_refs(aid, title, year)

            if details:
                all_results.extend(details)

            # 중간 저장
            if (i + 1) % SAVE_INTERVAL == 0:
                pd.DataFrame(all_results).to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

            time.sleep(0.5)

    except KeyboardInterrupt:
        print("\n[중단] 수집 중단됨 — 지금까지 수집한 데이터 저장합니다.")

    # 4. 최종 저장
    df_final = pd.DataFrame(all_results)
    df_final.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

    print(f"\n{'='*50}")
    print(f"  [완료] {OUTPUT_FILE}")
    print(f"  수집 논문 수: {df_final['source_id'].nunique()}건")
    print(f"  총 행 수 (인용 포함): {len(df_final)}행")
    print(f"{'='*50}")


if __name__ == "__main__":
    main()

[로딩] KCI_AI_논문_기본정보_목록_201601_202512.csv


FileNotFoundError: [Errno 2] No such file or directory: 'KCI_AI_논문_기본정보_목록_201601_202512.csv'

In [2]:
import pandas as pd
import html
import os

def fix_kci_categories(file_path):
    # 1. 파일 존재 확인 및 로드
    if not os.path.exists(file_path):
        print(f"❌ 에러: {file_path} 파일을 찾을 수 없습니다.")
        return

    print(f"📖 {file_path} 로딩 중...")
    df = pd.read_csv(file_path)

    # 2. '분류' 컬럼 기호 복구 (Unescape)
    # &gt; -> > , &amp; -> & 등으로 변환합니다.
    if '분류' in df.columns:
        print("🪄 HTML 엔티티 복구 중 (e.g., &gt; -> >) ...")
        
        # 주기적 저장
        if len(all_results) >= SAVE_INTERVAL:
            header = not os.path.exists(OUTPUT_FILE)
            pd.DataFrame(all_results).to_csv(OUTPUT_FILE, mode='a', index=False, header=header, encoding='utf-8-sig')
            all_results = [] 
            
        time.sleep(1.0) # API 부하 방지

except KeyboardInterrupt:
    print("\n🛑 중단되었습니다. 현재까지 데이터를 저장합니다.")

# 5. 최종 저장
if all_results:
    header = not os.path.exists(OUTPUT_FILE)
    pd.DataFrame(all_results).to_csv(OUTPUT_FILE, mode='a', index=False, header=header, encoding='utf-8-sig')

print(f"🏁 수집 완료! 파일 확인: {OUTPUT_FILE}")

📖 법학_AI_논문_상세정보_리스트.csv 로딩 중...
🪄 HTML 엔티티 복구 중 (e.g., &gt; -> >) ...
✅ 복구 완료!
--------------------------------------------------
📊 수정된 분류 데이터 샘플:
['사회과학 > 법학', '사회과학 > 법학', '사회과학 > 법학', '사회과학 > 법학', '사회과학 > 법학']
--------------------------------------------------
🏁 모든 작업이 완료되었습니다! 저장된 파일: 법학_AI_논문_상세정보_리스트.csv


### 테스트 코드

In [ ]:
import os
import requests
import pandas as pd
import xml.etree.ElementTree as ET
import time
import re  # 정규표현식 추가
from tqdm import tqdm
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("KCI_API_KEY")
BASE_URL = "https://open.kci.go.kr/po/openapi/openApiSearch.kci"

def fetch_references_ultimate(article_id):
    references = []
    params = {"apiCode": "articleDetail", "key": API_KEY, "id": article_id}
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Accept": "application/xml"
    }
    
    try:
        response = requests.get(BASE_URL, params=params, headers=headers, timeout=30)
        response.encoding = 'utf-8'
        raw_xml = response.text
        
        # 1. 기본 체크
        if "<total>0</total>" in raw_xml:
            return [], "데이터 없음(Total 0)"

        # 2. 정규표현식으로 데이터 직접 추출 (가장 강력한 우회 방법)
        # <reference ...> ... </reference> 사이의 내용을 모두 긁어옵니다.
        # CDATA 태그가 있어도 상관없이 매칭합니다.
        ref_matches = re.findall(r'<reference([^>]*)>(?:<!\[CDATA\[)?(.*?)(?:\]\]>)?</reference>', raw_xml, re.DOTALL)
        
        if ref_matches:
            for attrs, content in ref_matches:
                # 속성값에서 arti-id 추출 (예: arti-id="ART00123")
                arti_id_match = re.search(r'arti-id="([^"]*)"', attrs)
                type_name_match = re.search(r'type-name="([^"]*)"', attrs)
                
                references.append({
                    "source_id": article_id,
                    "target_arti_id": arti_id_match.group(1) if arti_id_match else "",
                    "ref_type": type_name_match.group(1) if type_name_match else "",
                    "raw_citation": content.strip()
                })
            return references, "성공"
        
        # 3. 만약 정규표현식마저 실패했다면 응답 내용 확인용 에러 메시지
        snippet = raw_xml.replace('\n', '')[:100]
        return [], f"추출 실패 (응답 앞부분: {snippet}...)"

    except Exception as e:
        return [], f"예외 발생: {str(e)}"

# --- 실행부 ---
INPUT_FILE = '전체_법학_논문목록.csv'
TEST_OUTPUT_FILE = '법학_인용_최종_우회결과.csv'

if not os.path.exists(INPUT_FILE):
    print("파일 없음")
else:
    df_articles = pd.read_csv(INPUT_FILE)
    df_2020 = df_articles[df_articles['발행연도'].astype(str) == '2020']
    
    # 확실히 있는 997번을 포함하여 11개 구성
    test_ids = ["ART002565997"] + [aid for aid in df_2020['논문ID'].unique() if aid != "ART002565997"]
    test_ids = test_ids[:11]

    all_results = []
    print(f"총 {len(test_ids)}개 논문 우회 테스트 시작")

    for aid in tqdm(test_ids):
        refs, status = fetch_references_ultimate(aid)
        if refs:
            all_results.extend(refs)
            print(f" ✅ ID {aid}: {len(refs)}건 수집 완료")
        else:
            print(f" ❌ ID {aid}: {status}")
        
        time.sleep(1.5)

    if all_results:
        pd.DataFrame(all_results).to_csv(TEST_OUTPUT_FILE, index=False, encoding='utf-8-sig')
        print(f"\n결과 저장 완료: {TEST_OUTPUT_FILE}")